# MIPIC on Kaggle — audited build

TinyBERT-6L, paper Table 7 settings: 5 epochs, lr `2e-5`, max length `256`,
**global batch 16**, cosine + 10% warmup, AdamW, `tau = 0.05`, `alpha = 0.4`,
PIC chain `(1,16) -> (2,32) -> (3,64) -> (4,256) -> (5,512) -> (6,768)`.
Corpus: the supplied `final_data.csv`, used whole (all 20,240 rows, `DGEM_FORMAT` kept).

**Run cells top to bottom.** Cell 6 picks the execution mode — read it before running.

| | |
| --- | --- |
| `RUN_MODE = "single"` | 1 GPU, per-device batch 16. Global batch 16 = paper-exact. No NCCL, so it cannot deadlock. **Default.** |
| `RUN_MODE = "ddp"` | 2xT4, 8/GPU x 2 = global batch 16. Faster, but this is the path that hung. |

Both modes write the same checkpoint and resume from each other, because both
produce 1,265 optimizer steps per epoch (20,240/16 on one rank; 10,120/8 on each
of two ranks). The scheduler state is therefore interchangeable.

## 1. Repository bootstrap

In [ ]:
from pathlib import Path
import os, sys, shutil, zipfile

work_root = Path("/kaggle/working")
current = Path.cwd()

if (current / "run_experiments.py").exists():
    repo_root = current
else:
    repo_root = work_root / "mipic"
    candidates = list(Path("/kaggle/input").rglob("run_experiments.py")) if Path("/kaggle/input").exists() else []
    if candidates:
        source = candidates[0].parent
        if repo_root.exists():
            shutil.rmtree(repo_root)
        shutil.copytree(source, repo_root,
                        ignore=shutil.ignore_patterns("results", "__pycache__", ".pytest_cache", "*.pyc"))
    else:
        zips = []
        for candidate in (Path("/kaggle/input").rglob("*.zip") if Path("/kaggle/input").exists() else []):
            try:
                with zipfile.ZipFile(candidate) as zf:
                    if any(n.endswith("run_experiments.py") for n in zf.namelist()):
                        zips.append(candidate)
            except zipfile.BadZipFile:
                continue
        if not zips:
            raise FileNotFoundError("MIPIC repository not found under /kaggle/input")
        if repo_root.exists():
            shutil.rmtree(repo_root)
        repo_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(repo_root)
        repo_root = next(iter(repo_root.rglob("run_experiments.py"))).parent

os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root)
print("Repository:", repo_root)
print("Version:", (repo_root / "VERSION.txt").read_text().splitlines()[0])

In [ ]:
!pip install -q -r requirements-kaggle.txt

## 2. Hardware

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator)."

## 3. Locate the dataset

Nothing is filtered or regenerated. Do **not** run `prepare_training_corpus.py`.

In [ ]:
import pandas as pd
from pathlib import Path

matches = sorted(Path("/kaggle/input").rglob("final_data.csv"))
if not matches:
    raise FileNotFoundError("Attach the Kaggle dataset containing final_data.csv (Add Input).")

# Point at the DIRECTORY. run_experiments.py resolves the three files it needs
# from it: final_data.csv (paper A.2 pooled corpus), tweet_train_3k.csv and
# tweet_test.csv (Table 6 probe splits). There is one configuration, no presets.
data_dir = matches[0].parent
os.environ["MIPIC_DATA_DIR"] = str(data_dir)
print("Data directory:", data_dir)

# tweet_train.csv is the PROBE train split (26,732 rows, full).
# tweet_train_3k.csv is the slice inside the contrastive corpus.
expected = ["final_data.csv", "tweet_train.csv", "tweet_test.csv", "tweet_train_3k.csv"]
for name in expected:
    found = data_dir / name
    if not found.exists():
        hits = sorted(Path("/kaggle/input").rglob(name)) or [Path(repo_root) / "data" / name]
        found = hits[0]
    status = "ok " if found.exists() else "MISSING"
    rows = len(pd.read_csv(found)) if found.exists() else 0
    print(f"  {status} {name:22} {rows:>6} rows   {found}")

frame = pd.read_csv(data_dir / "final_data.csv")
print("\nPooled corpus sources:")
print(frame["source"].value_counts().to_string())


## 4. Test gate

Never spend GPU quota before this passes. Expect **100 passed**.

In [ ]:
!python -m compileall -q . && echo "compileall OK"
!python -m pytest -q

## 5. Shape and gradient traces (CPU, seconds)

In [ ]:
!python tensor_traces.py

## 6. Execution mode and run configuration

`RUN_MODE = "single"` is the default: one GPU at per-device batch 16 is the same
global batch 16 the paper specifies, and it removes NCCL entirely.

Switch to `"ddp"` only if you want the ~2x speedup and are willing to hit the
hang again. The DDP mitigations below are all applied automatically in that mode.

In [ ]:
RUN_MODE = "single"          # "single" | "ddp"
OUTPUT_DIR = "/kaggle/working/mipic_results"
FRESH = False                # True DELETES any existing checkpoint. Keep False to resume.

os.environ["MIPIC_OUTPUT_DIR"] = OUTPUT_DIR
os.environ["MIPIC_FRESH"] = "1" if FRESH else "0"
os.environ["MIPIC_EPOCHS"] = "5"
os.environ["MIPIC_LEARNING_RATE"] = "2e-5"
os.environ["MIPIC_MAX_LENGTH"] = "256"
os.environ["MIPIC_AMP"] = "1"

# Everything else is hardcoded in the repository: the pooled corpus, the probe
# splits, the pooler read-out, SIA/PIC mean aggregation, gradient clipping at
# 1.0, alpha 0.4, tau 0.05. There are no configuration switches to set.

# Hang-resistance. DataLoader workers share Kaggle's small /dev/shm with the
# NCCL transport and fork a Rust tokenizer; both are documented deadlock
# sources, and tokenising short sentences is cheap next to SIA's 36 CKA terms.
os.environ["MIPIC_NUM_WORKERS"] = "0"
os.environ["MIPIC_LOG_EVERY"] = "50"            # every rank prints its own heartbeat
os.environ["MIPIC_STACK_DUMP_SECONDS"] = "900"  # each rank self-dumps its stack every 15 min

# Both branches set EVERY mode-specific variable, so re-running this cell after
# switching RUN_MODE fully overrides the previous mode. In particular
# CUDA_VISIBLE_DEVICES must be widened again for ddp: leaving it at "0" from a
# single-GPU run makes rank 1 fail with "invalid device ordinal".
if RUN_MODE == "ddp":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    os.environ["MIPIC_BATCH_SIZE"] = "8"         # 8/GPU x 2 = global 16
    os.environ["MIPIC_DDP_TIMEOUT_MIN"] = "20"   # a stalled collective raises instead of hanging
    os.environ["NCCL_P2P_DISABLE"] = "1"         # Kaggle T4 pairs have no NVLink
    os.environ["MIPIC_DDP_STATIC_GRAPH"] = "1"   # pin reducer/collective ordering
    # Escalate only if it still hangs:
    # os.environ["NCCL_SHM_DISABLE"] = "1"
    # os.environ["NCCL_DEBUG"] = "INFO"
    # os.environ["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
    LAUNCH = "torchrun --standalone --nproc_per_node=2 run_experiments.py"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["MIPIC_BATCH_SIZE"] = "16"        # single device, global 16
    os.environ.pop("MIPIC_DDP_STATIC_GRAPH", None)
    os.environ.pop("MIPIC_DDP_TIMEOUT_MIN", None)
    os.environ.pop("NCCL_P2P_DISABLE", None)
    LAUNCH = "python run_experiments.py"

# torch.cuda.device_count() is cached from the first CUDA init in THIS kernel,
# so it does not reflect CUDA_VISIBLE_DEVICES set afterwards. Ask a fresh
# subprocess, which is what torchrun will actually launch.
import subprocess
visible = int(subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
    capture_output=True, text=True, env=os.environ,
).stdout.strip() or 0)
need = 2 if RUN_MODE == "ddp" else 1
assert visible >= need, (
    f"RUN_MODE={RUN_MODE!r} needs {need} GPU(s); a subprocess sees {visible} "
    f"with CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']!r}. "
    "Restart the kernel if you changed the accelerator setting."
)

print("mode:", RUN_MODE)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"], "-> subprocess sees", visible, "GPU(s)")
print("per-device batch:", os.environ["MIPIC_BATCH_SIZE"],
      "| global batch:", int(os.environ["MIPIC_BATCH_SIZE"]) * (2 if RUN_MODE == "ddp" else 1))
print("output:", OUTPUT_DIR, "| fresh:", FRESH)
print("launch:", LAUNCH)

## 7. Resume status

Completed epochs are never lost. Check before launching.

In [ ]:
state_path = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "training_state.pt"
done_path  = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "model.pt"
if state_path.exists():
    saved = torch.load(state_path, map_location="cpu")
    print(f"Resumable checkpoint: epoch {saved['epoch']} of 5")
    for i, h in enumerate(saved.get("history", []), 1):
        print("  epoch", i, {k: round(v, 4) for k, v in h.items()})
    if FRESH:
        print("\n!! FRESH=True will DELETE this. Set FRESH=False and re-run cell 6.")
elif done_path.exists():
    print("Completed checkpoint present:", done_path, done_path.stat().st_size, "bytes")
else:
    print("No checkpoint; will start from scratch.")

## 8. Pipeline check (about a minute) - NOT a result

Truncated training that still exercises `train -> save -> assert non-empty ->
evaluate`, so a path or checkpoint problem surfaces now rather than after hours
of GPU time.

It trains on **48 sentences** (3 batches x 8/rank x 2 ranks), i.e. 0.24% of one
epoch, and probes 300/300 rows. The run prints a `TRUNCATED RUN` banner and a
`PLUMBING CHECK ONLY` warning over its results table, and stamps
`truncated_run: true` into its results JSON. **Ignore the numbers.** Writes to a
throwaway directory.

In [ ]:
check_env = {
    "MIPIC_MAX_BATCHES": "3",
    "MIPIC_SMOKE_FULL_PIPELINE": "1",
    "MIPIC_EPOCHS": "1",
    "MIPIC_EVAL_TRAIN_SAMPLES": "300",
    "MIPIC_EVAL_TEST_SAMPLES": "300",
    "MIPIC_FRESH": "1",
    "MIPIC_OUTPUT_DIR": "/kaggle/working/mipic_pipeline_check",
}
prefix = " ".join(f"{k}={v}" for k, v in check_env.items())
!{prefix} {LAUNCH} --model mipic

## 9. Full 5-epoch run - this is the real one

Heartbeats look like `[rank 0] 22:31:04 epoch 3/5: batch 550/1265`. Confirm you
see **`epoch 1/5`** and rising batch counts, and that no `TRUNCATED RUN` banner
appears. Expect ~13 minutes per arm on 2xT4.

If output stops for more than ~20 minutes, go to cell 11 **before** interrupting.

Each epoch prints a held-out probe score:

```
Epoch 1: held-out probe macro-F1 63.10  <- best so far, checkpointed
Epoch 2: held-out probe macro-F1 64.80  <- best so far, checkpointed
Epoch 3: held-out probe macro-F1 61.20
```

That probe is fitted on 80% of the probe-TRAIN rows and scored on the held-out
20%; the test split is never touched. The best epoch is kept in
`checkpoints/<arm>/best.pt` and evaluated alongside the final epoch, which
appears as `<arm>_last_epoch` in the results.

In [ ]:
!{LAUNCH} --model mipic

## 10. Results

In [ ]:
import json
from pathlib import Path

results = Path(OUTPUT_DIR) / "tweeteval_results.json"
if results.exists():
    data = json.loads(results.read_text())
    print(json.dumps(data.get("metadata", {}), indent=1))
    for name, history in data.get("training_history", {}).items():
        print("\n" + name, "training history:")
        for i, h in enumerate(history, 1):
            print("  epoch", i, {k: round(v, 4) for k, v in h.items()})
    rows = []
    for name, dims in data.get("evaluations", {}).items():
        for dim, m in sorted(dims.items(), key=lambda kv: int(kv[0])):
            rows.append({"Model": name, "Dim": int(dim),
                         "Acc": round(m["accuracy"] * 100, 2),
                         "Macro_F1": round(m["f1"] * 100, 2),
                         "probe_iters": m.get("probe_n_iter"),
                         "converged": m.get("probe_converged")})
    display(pd.DataFrame(rows))
    if data.get("evaluation_errors"):
        print("\nEVALUATION ERRORS:", data["evaluation_errors"])
else:
    print("No results yet:", results)

for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print(f"{p}  ({p.stat().st_size:,} bytes)")

## 11. If the run hangs

Output stops mid-epoch, both processes alive, near-zero CPU/GPU.

`py-spy` will **not** work here — Kaggle's container blocks `ptrace`
(`Permission denied (os error 13)`). The cell below reads `/proc` instead, which
is always permitted. Run it from a **second notebook session** while the run is
still stuck, or from this one if the training cell is in a different session.

In [ ]:
!python diagnose_hang.py --sample-seconds 5

Read the output:

| Observation | Diagnosis | Action |
| --- | --- | --- |
| Every thread ~0% CPU, state `S`, WCHAN `futex_wait` / `pipe_read` | waiting on another process | `MIPIC_NUM_WORKERS=0` (already default), then `NCCL_SHM_DISABLE=1` |
| Threads pinned near 100%, state `R` | NCCL collective spinning for a peer that never posted | `MIPIC_DDP_STATIC_GRAPH=1` (already default in ddp mode), then `NCCL_SHM_DISABLE=1` |
| State `D`, low disk in the DISK section | blocked on I/O | free space in `/kaggle/working` |
| Only one process listed | the peer rank died | scroll the training log up for its exception |

Then set `RUN_MODE = "single"` in cell 6, confirm cell 7 shows the resumable
epoch, and re-run cell 9. Single-GPU mode is paper-exact on global batch size
and cannot deadlock.

## 12. Evaluation only, from an explicit checkpoint

Path-independent; does not require a training run in the same session.

In [ ]:
ckpt = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "model.pt"
if ckpt.exists():
    !MIPIC_CHECKPOINT={ckpt} python run_experiments.py --model mipic --eval-only
else:
    print("No completed checkpoint at", ckpt)

## 13. Optional: all four controlled conditions

`mrl`, `mrl_sia`, `mrl_pic`, `mipic` — identical corpus, batch size, schedule,
clipping and representation, so the arms are directly comparable. Roughly 4x the
runtime. `--skip-existing` resumes a partial sweep.

In [ ]:
# MIPIC_OUTPUT_DIR=/kaggle/working/mipic_results_all
# !MIPIC_OUTPUT_DIR=/kaggle/working/mipic_results_all {LAUNCH} --model all --skip-existing